# Exploratory Analysis of Engineered Stock Features

This notebook explores `data/processed/stock_features.csv`, the feature-engineered stock dataset produced by `src/feature_engineering.py`. It creates a concise dataset overview and saves each exploratory chart as an individual Matplotlib figure under `reports/figures/`.


## 1. Setup

Import analysis libraries, define project-relative paths, and create the figures output directory. The path logic works whether the notebook is run from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "stock_features.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

DATA_PATH, FIGURES_DIR


## 2. Load Feature-Engineered Dataset

Load the engineered feature file and parse the `date` column for time-series analysis.


In [ ]:
stocks = pd.read_csv(DATA_PATH)
stocks["date"] = pd.to_datetime(stocks["date"])
stocks = stocks.sort_values(["symbol", "date"]).reset_index(drop=True)

stocks.head()


## 3. Dataset Overview After Feature Engineering

Review the shape, date range, symbols, column types, missing values, and symbol-level coverage of the feature-engineered dataset.


In [ ]:
print(f"Rows: {stocks.shape[0]:,}")
print(f"Columns: {stocks.shape[1]:,}")
print(f"Date range: {stocks['date'].min().date()} to {stocks['date'].max().date()}")
print(f"Symbols: {', '.join(sorted(stocks['symbol'].unique()))}")

print("\nColumn dtypes:")
display(stocks.dtypes.rename("dtype").to_frame())

print("\nMissing values by column:")
display(stocks.isna().sum().rename("missing_values").to_frame())

print("\nSymbol-level coverage:")
symbol_overview = (
    stocks.groupby("symbol")
    .agg(
        row_count=("date", "size"),
        start_date=("date", "min"),
        end_date=("date", "max"),
        mean_close=("close", "mean"),
        mean_daily_return=("daily_return", "mean"),
        daily_return_std=("daily_return", "std"),
    )
    .sort_index()
)
display(symbol_overview)

print("\nNumeric summary:")
display(stocks.describe(include=[np.number]).T)


## 4. Closing Price Over Time by Symbol

Plot the daily closing price for each symbol to compare price history across the engineered dataset.


In [ ]:
fig = plt.figure()
for symbol, symbol_data in stocks.groupby("symbol"):
    plt.plot(symbol_data["date"], symbol_data["close"], label=symbol)

plt.title("Closing Price Over Time by Symbol")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend(title="Symbol")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "closing_price_by_symbol.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Cumulative Return Over Time by Symbol

Calculate and plot compounded cumulative returns from `daily_return` for each symbol.


In [ ]:
cumulative_returns = stocks.copy()
cumulative_returns["cumulative_return"] = (
    cumulative_returns.groupby("symbol")["daily_return"]
    .transform(lambda returns: (1 + returns).cumprod() - 1)
)

fig = plt.figure()
for symbol, symbol_data in cumulative_returns.groupby("symbol"):
    plt.plot(symbol_data["date"], symbol_data["cumulative_return"], label=symbol)

plt.title("Cumulative Return Over Time by Symbol")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend(title="Symbol")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "cumulative_return_by_symbol.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Daily Return Distribution by Symbol

Compare the distribution of `daily_return` values across symbols using one overlaid Matplotlib histogram figure.


In [ ]:
fig = plt.figure()
for symbol, symbol_data in stocks.groupby("symbol"):
    plt.hist(
        symbol_data["daily_return"].dropna(),
        bins=50,
        alpha=0.45,
        density=True,
        label=symbol,
    )

plt.title("Daily Return Distribution by Symbol")
plt.xlabel("Daily Return")
plt.ylabel("Density")
plt.legend(title="Symbol")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "daily_return_distribution_by_symbol.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Trading Volume Over Time by Symbol

Plot daily trading volume for each symbol to inspect volume patterns over time.


In [ ]:
fig = plt.figure()
for symbol, symbol_data in stocks.groupby("symbol"):
    plt.plot(symbol_data["date"], symbol_data["volume"], label=symbol)

plt.title("Trading Volume Over Time by Symbol")
plt.xlabel("Date")
plt.ylabel("Trading Volume")
plt.legend(title="Symbol")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "trading_volume_by_symbol.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Rolling 20-Day Volatility Over Time by Symbol

Plot the engineered `rolling_volatility_20` feature for each symbol to compare short-term volatility patterns.


In [ ]:
fig = plt.figure()
for symbol, symbol_data in stocks.groupby("symbol"):
    plt.plot(symbol_data["date"], symbol_data["rolling_volatility_20"], label=symbol)

plt.title("Rolling 20-Day Volatility Over Time by Symbol")
plt.xlabel("Date")
plt.ylabel("Rolling 20-Day Volatility")
plt.legend(title="Symbol")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "rolling_20_day_volatility_by_symbol.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Correlation Heatmap of Daily Returns by Symbol

Create a Matplotlib heatmap from the pairwise correlation matrix of daily returns by symbol.


In [ ]:
returns_wide = stocks.pivot(index="date", columns="symbol", values="daily_return")
return_correlations = returns_wide.corr()

fig = plt.figure()
image = plt.imshow(return_correlations, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(image, label="Correlation")
plt.xticks(range(len(return_correlations.columns)), return_correlations.columns, rotation=45, ha="right")
plt.yticks(range(len(return_correlations.index)), return_correlations.index)

for row_idx, symbol_row in enumerate(return_correlations.index):
    for col_idx, symbol_col in enumerate(return_correlations.columns):
        correlation = return_correlations.loc[symbol_row, symbol_col]
        plt.text(col_idx, row_idx, f"{correlation:.2f}", ha="center", va="center", color="black")

plt.title("Correlation Heatmap of Daily Returns by Symbol")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "daily_return_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Risk-Return Scatterplot

Annualize each symbol's average daily return and daily return volatility with a 252-trading-day convention, then plot annualized volatility against annualized return.


In [ ]:
TRADING_DAYS_PER_YEAR = 252

risk_return = (
    stocks.groupby("symbol")["daily_return"]
    .agg(mean_daily_return="mean", daily_volatility="std")
    .assign(
        annualized_return=lambda df: df["mean_daily_return"] * TRADING_DAYS_PER_YEAR,
        annualized_volatility=lambda df: df["daily_volatility"] * np.sqrt(TRADING_DAYS_PER_YEAR),
    )
    .sort_index()
)

fig = plt.figure()
plt.scatter(
    risk_return["annualized_volatility"],
    risk_return["annualized_return"],
    s=100,
)

for symbol, row in risk_return.iterrows():
    plt.annotate(
        symbol,
        (row["annualized_volatility"], row["annualized_return"]),
        textcoords="offset points",
        xytext=(6, 6),
    )

plt.title("Annualized Risk-Return by Symbol")
plt.xlabel("Annualized Volatility")
plt.ylabel("Annualized Return")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "annualized_risk_return_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

risk_return


## EDA Takeaways

- TODO: Add observations about the dataset coverage after running the notebook.
- TODO: Add observations about closing price and cumulative return patterns after running the notebook.
- TODO: Add observations about daily return distributions after running the notebook.
- TODO: Add observations about trading volume and rolling volatility after running the notebook.
- TODO: Add observations about return correlations and risk-return positioning after running the notebook.
